<a href="https://colab.research.google.com/github/peteparker123/graph-rag/blob/main/graph_rag_with_llmgraph_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GraphRAG using LLMGraphTransformer Approach

## This notebook demonstrates GraphRAG using LangChain's LLMGraphTransformer approach

In [ ]:
# Install required dependencies
!pip install --upgrade --quiet json-repair networkx langchain-core langchain-experimental langchain-community langchain

In [ ]:
# Install Google Generative AI for LLM and embeddings
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.8 MB/s eta 0:00:00


In [ ]:
# Install PDF loader
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Document loading and processing
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# LLM and Embeddings
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

# Graph transformation and retrieval
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain_classic.chains import GraphQAChain

# Graph libraries
import networkx as nx

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# ============================================================
# 2. LOAD ENVIRONMENT VARIABLES AND INITIALIZE LLM
# ============================================================

# Initialize Gemini LLM for graph transformation
# Using a more capable model for better knowledge graph extraction
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key="your gemini api key",
    temperature=0,  # Low temperature for consistent extraction
)

print("LLM initialized successfully!")

LLM initialized successfully!


In [ ]:
# ============================================================
# 3. LOAD PDF DOCUMENT
# ============================================================

# Update the PDF path to your document
pdf_path = "/content/resume_finale.pdf"  # Change this to your PDF path

try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} PDF pages")
except FileNotFoundError:
    print(f"PDF not found at {pdf_path}")
    print("Using sample text for demonstration...")
    # Sample text for demonstration
    sample_text = """
    Marie Curie, born in 1867, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
    She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
    Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
    She was, in 1906, the first woman to become a professor at the University of Paris.
    """
    documents = [Document(page_content=sample_text)]

Loaded 1 PDF pages


In [ ]:
# ============================================================
# 4. SPLIT DOCUMENTS INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # Larger chunks for better context
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")
print(f"\nFirst chunk preview:")
print(chunks[0].page_content[:500])

Created 2 chunks

First chunk preview:
JAI AKASH 
github.com/peteparker123 | huggingface.co/peteparker456 | linkedin.com/in/jai-akash-a3b7a0272 
PROFILE 
Fourth-year Electronics & Computer Science Engineering student passionate about building innovative AI systems, 
conducting research, and solving real-world problems. Interested in Large Language Models, Deep Learning, and 
Agentic AI, with hands-on experience developing transformer-based models, RAG pipelines, and intelligent 
applications. 
EDUCATION 
B. Tech in Electronics & Comp


In [ ]:
# ============================================================
# 5. INITIALIZE LLMGraphTransformer
# ============================================================

# Define allowed node types for your domain
# Customize these based on your document type
allowed_nodes = [
    "Person",
    "EducationalInstitution",
    "Degree",
    "Skill",
    "Technology",
    "Project",
    "ResearchTopic",
    "Achievement",
    "ApplicationDomain",
]
# Define allowed relationships for your domain
allowed_relationships = [
    "STUDIED_AT",
    "PURSUING",
    "HAS_SKILL",
    "WORKED_ON",
    "USES",
    "APPLIES",
    "RESEARCHED",
    "ACHIEVED",
]

# Create LLMGraphTransformer with filtered nodes and relationships
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    strict_mode=True  # Allow some flexibility in extraction
)

print("LLMGraphTransformer initialized successfully!")
print(f"\nAllowed Nodes: {allowed_nodes}")
print(f"\nAllowed Relationships: {allowed_relationships}")

LLMGraphTransformer initialized successfully!

Allowed Nodes: ['Person', 'EducationalInstitution', 'Degree', 'Skill', 'Technology', 'Project', 'ResearchTopic', 'Achievement', 'ApplicationDomain']

Allowed Relationships: ['STUDIED_AT', 'PURSUING', 'HAS_SKILL', 'WORKED_ON', 'USES', 'APPLIES', 'RESEARCHED', 'ACHIEVED']


In [ ]:
# ============================================================
# 6. CONVERT DOCUMENTS TO GRAPH DOCUMENTS
# ============================================================

print("Converting documents to graph documents...\n")

all_graph_documents = []

for index, chunk in enumerate(chunks):
    print(f"Processing chunk {index + 1}/{len(chunks)}...")

    try:
        # Convert chunk to graph documents
        graph_documents = llm_transformer.convert_to_graph_documents([chunk])
        all_graph_documents.extend(graph_documents)

        # Display extracted information
        if graph_documents:
            print(f"  ✓ Extracted {len(graph_documents[0].nodes)} nodes and {len(graph_documents[0].relationships)} relationships")
        else:
            print(f"  ⚠ No graph documents extracted from this chunk")

    except Exception as e:
        print(f"  ✗ Error processing chunk {index + 1}: {str(e)}")
        continue

print(f"\n✓ Conversion complete! Total graph documents created: {len(all_graph_documents)}")

Converting documents to graph documents...

Processing chunk 1/2...
  ✓ Extracted 42 nodes and 50 relationships
Processing chunk 2/2...
  ✓ Extracted 17 nodes and 16 relationships

✓ Conversion complete! Total graph documents created: 2


In [ ]:
# ============================================================
# 7. DISPLAY EXTRACTED GRAPH INFORMATION
# ============================================================

print("\n" + "="*80)
print("EXTRACTED GRAPH INFORMATION")
print("="*80)

all_nodes = []
all_relationships = []

for doc_index, graph_doc in enumerate(all_graph_documents):
    print(f"\n--- Graph Document {doc_index + 1} ---")

    # Extract and display nodes
    print(f"\nNodes ({len(graph_doc.nodes)}):")
    for node in graph_doc.nodes:
        print(f"  • {node.id} ({node.type})")
        all_nodes.append((node.id, node.type))

    # Extract and display relationships
    print(f"\nRelationships ({len(graph_doc.relationships)}):")
    for rel in graph_doc.relationships:
        print(f"  • {rel.source.id} --[{rel.type}]--> {rel.target.id}")
        all_relationships.append((rel.source.id, rel.type, rel.target.id))

    print("-" * 80)

print(f"\nTotal unique nodes: {len(set(all_nodes))}")
print(f"Total relationships: {len(all_relationships)}")


EXTRACTED GRAPH INFORMATION

--- Graph Document 1 ---

Nodes (42):
  • Jai Akash (Person)
  • Amrita School Of Engineering, Bangalore (Educationalinstitution)
  • B. Tech In Electronics & Computer Science Engineering (Degree)
  • Python (Skill)
  • C (Skill)
  • Pytorch (Technology)
  • Tensorflow (Technology)
  • Transformers (Technology)
  • Scikit-Learn (Technology)
  • Opencv (Technology)
  • Ultralytics (Technology)
  • Langchain (Technology)
  • Langgraph (Technology)
  • Llamaindex (Technology)
  • Faiss (Technology)
  • Chromadb (Technology)
  • Hugging Face (Technology)
  • Streamlit (Technology)
  • Gradio (Technology)
  • N8N (Technology)
  • Wandb (Technology)
  • Pandas (Technology)
  • Numpy (Technology)
  • Librosa (Technology)
  • Nltk (Technology)
  • Research-Oriented Thinking (Skill)
  • Problem Solving (Skill)
  • Open-Source Contribution (Skill)
  • Team Leadership (Skill)
  • Large Language Models (Researchtopic)
  • Deep Learning (Researchtopic)
  • Agentic Ai (

In [ ]:
# ============================================================
# 8. BUILD NETWORKX KNOWLEDGE GRAPH
# ============================================================

print("Building NetworkX knowledge graph...\n")

# Create a directed graph
graph = NetworkxEntityGraph()

# Add all nodes from all graph documents
for graph_doc in all_graph_documents:
    for node in graph_doc.nodes:
        graph.add_node(node.id)

# Add all relationships as edges
for graph_doc in all_graph_documents:
    for edge in graph_doc.relationships:
        graph._graph.add_edge(
            edge.source.id,
            edge.target.id,
            relation=edge.type,
        )

print(f"✓ Knowledge Graph created successfully!")
print(f"  • Total Nodes: {graph._graph.number_of_nodes()}")
print(f"  • Total Edges: {graph._graph.number_of_edges()}")

Building NetworkX knowledge graph...

✓ Knowledge Graph created successfully!
  • Total Nodes: 56
  • Total Edges: 66


In [ ]:
# ============================================================
# 9. VISUALIZE THE KNOWLEDGE GRAPH (Optional)
# ============================================================

import matplotlib.pyplot as plt

print("Generating knowledge graph visualization...\n")

# Only visualize if graph is not too large
if graph._graph.number_of_nodes() <= 20:
    plt.figure(figsize=(15, 10))

    # Use spring layout for better visualization
    pos = nx.spring_layout(
        graph._graph,
        k=2,
        iterations=50,
        seed=42
    )

    # Draw nodes
    nx.draw_networkx_nodes(
        graph._graph,
        pos,
        node_color='lightblue',
        node_size=2000,
        alpha=0.9
    )

    # Draw edges
    nx.draw_networkx_edges(
        graph._graph,
        pos,
        edge_color='gray',
        arrows=True,
        arrowsize=20,
        arrowstyle='->',
        connectionstyle='arc3,rad=0.1'
    )

    # Draw labels
    nx.draw_networkx_labels(
        graph._graph,
        pos,
        font_size=8,
        font_weight='bold'
    )

    # Draw edge labels (relationship types)
    edge_labels = nx.get_edge_attributes(graph._graph, 'relation')
    nx.draw_networkx_edge_labels(
        graph._graph,
        pos,
        edge_labels,
        font_size=7
    )

    plt.title("Knowledge Graph Visualization", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

else:
    print(f"⚠ Graph too large ({graph._graph.number_of_nodes()} nodes) for visualization")
    print(f"  You can still query it using the GraphQAChain")

Generating knowledge graph visualization...

⚠ Graph too large (56 nodes) for visualization
  You can still query it using the GraphQAChain


In [ ]:
# ============================================================
# 10. CREATE GRAPH QA CHAIN
# ============================================================

print("Creating GraphQAChain...\n")

# Create the chain that will answer questions using the knowledge graph
chain = GraphQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True
)

print("✓ GraphQAChain created successfully!")

Creating GraphQAChain...

✓ GraphQAChain created successfully!


In [ ]:
# ============================================================
# 12. INTERACTIVE QUERY INTERFACE
# ============================================================

print("\n" + "="*80)
print("INTERACTIVE KNOWLEDGE GRAPH QUERY")
print("="*80)
print("\nEnter your questions about the document. Type 'quit' to exit.\n")

while True:
    question = input("\n🔍 Ask a question: ").strip()

    if question.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Exiting knowledge graph query interface...")
        break

    if not question:
        print("⚠ Please enter a valid question.")
        continue

    print("\n" + "-"*80)
    try:
        response = chain.run(question)
        print(f"\n📝 Answer:\n{response}")
    except Exception as e:
        print(f"❌ Error: {str(e)}")

    print("-"*80)


INTERACTIVE KNOWLEDGE GRAPH QUERY

Enter your questions about the document. Type 'quit' to exit.


🔍 Ask a question: What evidence shows that Jai Akash has experience with both transformer models and real-world AI applications?

--------------------------------------------------------------------------------


> Entering new GraphQAChain chain...
Entities Extracted:
Jai Akash
Full Context:
Jai Akash STUDIED_AT Amrita School Of Engineering, Bangalore
Jai Akash PURSUING B. Tech In Electronics & Computer Science Engineering
Jai Akash HAS_SKILL Python
Jai Akash HAS_SKILL C
Jai Akash HAS_SKILL Pytorch
Jai Akash HAS_SKILL Tensorflow
Jai Akash HAS_SKILL Transformers
Jai Akash HAS_SKILL Scikit-Learn
Jai Akash HAS_SKILL Opencv
Jai Akash HAS_SKILL Ultralytics
Jai Akash HAS_SKILL Langchain
Jai Akash HAS_SKILL Langgraph
Jai Akash HAS_SKILL Llamaindex
Jai Akash HAS_SKILL Faiss
Jai Akash HAS_SKILL Chromadb
Jai Akash HAS_SKILL Hugging Face
Jai Akash HAS_SKILL Streamlit
Jai Akash HAS_SKILL Gradio
Jai

In [ ]:
# ============================================================
# 13. GRAPH ANALYSIS AND STATISTICS
# ============================================================

print("\n" + "="*80)
print("KNOWLEDGE GRAPH ANALYSIS")
print("="*80 + "\n")

# Basic statistics
print("Graph Statistics:")
print(f"  • Number of Nodes: {graph._graph.number_of_nodes()}")
print(f"  • Number of Edges: {graph._graph.number_of_edges()}")
print(f"  • Graph Density: {nx.density(graph._graph):.4f}")

# Node degree analysis
print("\nTop 10 Most Connected Nodes (by degree):")
degree_dict = dict(graph._graph.degree())
sorted_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]

for node, degree in sorted_degree:
    print(f"  • {node}: {degree} connections")

# Edge type analysis
print("\nRelationship Type Distribution:")
edge_types = {}
for source, target, data in graph._graph.edges(data=True):
    rel_type = data.get('relation', 'UNKNOWN')
    edge_types[rel_type] = edge_types.get(rel_type, 0) + 1

for rel_type, count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True):
    print(f"  • {rel_type}: {count}")


KNOWLEDGE GRAPH ANALYSIS

Graph Statistics:
  • Number of Nodes: 56
  • Number of Edges: 66
  • Graph Density: 0.0214

Top 10 Most Connected Nodes (by degree):
  • Jai Akash: 37 connections
  • Applicant: 9 connections
  • Tamil Gpt (40M Parameters): 6 connections
  • Python: 5 connections
  • Legal Ai Chatbot (Langchain): 5 connections
  • Classmind – Real-Time Ai Notes: 5 connections
  • Chat Epstein – Rag Agent: 5 connections
  • Langchain: 3 connections
  • Pytorch: 2 connections
  • Transformers: 2 connections

Relationship Type Distribution:
  • HAS_SKILL: 29
  • USES: 15
  • RESEARCHED: 6
  • WORKED_ON: 6
  • ACHIEVED: 5
  • APPLIES: 3
  • STUDIED_AT: 1
  • PURSUING: 1


In [1]:
import json
from pathlib import Path

def remove_widget_metadata(input_file, output_file=None):
    """
    Removes widget metadata from a Jupyter notebook if it exists.

    Parameters:
        input_file (str): Path to the input .ipynb file
        output_file (str): Path to save the cleaned notebook.
                           If None, overwrites the original file.
    """

    input_file = Path(input_file)

    if output_file is None:
        output_file = input_file
    else:
        output_file = Path(output_file)

    # Load notebook
    with open(input_file, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    modified = False

    # Remove notebook-level widget metadata
    if "metadata" in notebook and "widgets" in notebook["metadata"]:
        del notebook["metadata"]["widgets"]
        modified = True
        print("✓ Removed notebook metadata.widgets")

    # Remove widget metadata from cells (if present)
    for cell in notebook.get("cells", []):
        if "metadata" in cell and "widgets" in cell["metadata"]:
            del cell["metadata"]["widgets"]
            modified = True

    # Save notebook
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(notebook, f, indent=1, ensure_ascii=False)

    if modified:
        print(f"\nCleaned notebook saved to:\n{output_file}")
    else:
        print("\nNo widget metadata found. Notebook was unchanged.")

# Example
remove_widget_metadata(
    input_file="/content/graph_rag_llm_transformer_(1).ipynb",
    output_file="graph_rag_with_llmgraph_transformer.ipynb"
)


No widget metadata found. Notebook was unchanged.
